In [ ]:
#%load_ext autoreload
#%autoreload 2

<a name="top"></a>
# <center>Greenland ICESHEET Simulator</center>

## Abstract

- This Jupyter Notebook tool implements an easy-to-use set up to run the [ICESHEET](https://doi.org/10.5194/gmd-9-1673-2016) program with the Greenland Ice Sheet, for educational purposes.

- This tool was ported from the GitHub [Greenland_ICESHEET](https://github.com/evangowan/Greenland_ICESHEET) repository, developed by Evan James Gowan.


## Overview

- This tool enables you to run ICESHEET simulations for the Greenland Ice Sheet using your selected simulation parameters.

- The output of this tool are PDF files containing the Greenland ICESHEET simulation results.

<a name="userguide"></a>
## User Guide

1. [Enter the run parameters and run a simulation](#step_1)<br />
2. [View the simulation results](#step_2)<br />
3. [View the log output](#step_3)<br />


In [ ]:
# As of 09/2024, tested with the Jupyter Notebook (202210) tool and the geospatial-plus python3

# Setup and preoprocessing:

import atexit
import datetime
import getpass
import glob
import math
import numpy as np
import os
import pdf2image
import pandas as pd
import platform
import shutil
import subprocess
import sys
import time

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, clear_output, Javascript, IFrame

from PIL import Image as PIL_image
from PIL import ImageChops as PIL_image_chops
from IPython.display import Image as IPython_display_image

import matplotlib.pyplot as plt
import matplotlib.image as plt_image
#from mpl_toolkits.mplot3d import Axes3D
import matplotlib.backends.backend_pdf

# Apparently for the default font
# the minus sign is missing so use a hyphen instead
# so hyphen will show up in the saved pdf
plt.rcParams['axes.unicode_minus'] = False

import hublib
#print (help(hublib))
import hublib.ui as ui
#print (help(ui))
#import hublib.use
#print (help(hublib.use))

#print(sys.path)

Tooldir = os.path.dirname(os.path.realpath(" "))
#print ('Tooldir: ', Tooldir)

# Add to PYTHONPATH
sys.path.insert (1, os.path.join(Tooldir, "bin"))

User = getpass.getuser()

# Downloaded hublib.ui.download.py from https://pypi.org/project/hublib/0.9.97/#files.\n",
from download import Download

# Configuration parameters

np.set_printoptions(threshold=np.inf) 

self_log_filepath = os.path.join(Tooldir, 'icesheet1_log_file.txt')
self_log_backup_filepath = os.path.join(Tooldir, 'icesheet1_log_backup_file.txt')

widget_border_style = '1px solid black'
widget_output_border_style = '1px solid black'

BOLD = '\033[1m'
SUCCESS = '\033[92m'
WARNING = '\033[93m'
FAIL = '\033[91m'
END = '\033[0m'

GREENLAND_ICESHEET_FILES = ['icesheet1.ipynb', 'LICENSE', 'plot.sh', 'projection_info.sh','readme.txt','README.md', 'run_icesheet.sh','settings.sh', 'icesheet1_log_file.txt']


In [ ]:
def clean_up_generated_files():
    
    for filename in os.listdir(Tooldir):
        filepath = os.path.join(Tooldir, filename)
        if os.path.isfile(filepath) and filename not in GREENLAND_ICESHEET_FILES:
            #print ('Deleting %s ' %filepath)
            os.remove(filepath)
    
    dirpath = os.path.join(Tooldir, 'plots')
    if os.path.exists(dirpath) and os.path.isdir(dirpath):
        shutil.rmtree(dirpath)
         
    dirpath = os.path.join(Tooldir, 'contours')
    if os.path.exists(dirpath) and os.path.isdir(dirpath):
        shutil.rmtree(dirpath)
         
    dirpath = os.path.join(Tooldir, 'margins')
    if os.path.exists(dirpath) and os.path.isdir(dirpath):
        shutil.rmtree(dirpath)
         
    dirpath = os.path.join(Tooldir, 'ice_thickness')
    if os.path.exists(dirpath) and os.path.isdir(dirpath):
        shutil.rmtree(dirpath)


In [ ]:
#clean_up_generated_files()

In [ ]:
# Clean up: remove files from the data/results folder and the bin/__pycache__ folder
def exit_handler():
    
    # Free up disk space
    clean_up_generated_files()
         
    FH1.flush()
    FH1.close()

atexit.register(exit_handler);   

In [ ]:
# prevent In[] and Out[] from displaying on left
#HTML('''
#<style>.prompt{width: 0px; min-width: 0px; visibility: collapse}</style>
#''')

In [ ]:
#https://api.jquery.com/ready/
HTML('''
<script>
    function scroll_to_top() {
        Jupyter.notebook.scroll_to_top();
    } 
    $( window ).on( "load", scroll_to_top() );
</script>
''')

In [ ]:
# Button styles
HTML('''
<style>.buttontextclass { color:black ; font-size:130%}</style>
<style>.resetbuttontextclass { color:black ; font-size:90%}</style>
''')

In [ ]:
# Center displays in notebook
HTML("""
<style>
.output_png {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")

In [ ]:
#https://stackoverflow.com/questions/36757301/disable-ipython-notebook-autoscrolling

In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}


In [ ]:
# LD_LIBRARY_PATH is null
#print(os.environ['PATH'])
#if 'LD_LIBRARY_PATH' in os.environ:
    #print(os.environ['LD_LIBRARY_PATH'])


In [ ]:
# Initialize

BUTTON_WIDTH = '250px'
BUTTON_HEIGHT = '30px'

if os.path.exists(self_log_filepath):
    shutil.move (self_log_filepath, self_log_backup_filepath)
    
FH1 = open(self_log_filepath, 'w')

# Utility Functions

def disable_widgets():
    
    run_simulation_button.disabled = True
    show_log_output_button.disabled = True
    downloadTXTButton.disabled = True


def enable_widgets():
    
    run_simulation_button.disabled = False
    show_log_output_button.disabled = False
    downloadTXTButton.disabled = False

def log_info (message):
    
    FH1.write('%s\n' %message)
    FH1.flush()

def log_status (output_widget, message):
    
    with output_widget:
        print (message)
    log_info (message)
    
def log_success (output_widget, message):
    
    with output_widget:
        print ('%s%s%s' %(SUCCESS,message,END))
    log_info (message)
    
def log_warning (output_widget, message):
    
    with output_widget:
        print ('%s%s%s' %(WARNING,message,END))
    log_info (message)
    
def log_error (output_widget, message):
    
    with output_widget:
        print ('%s%s%s' %(FAIL,message,END))
    log_info (message)
    


In [ ]:
output_widget = widgets.Output(layout={'border': '1px solid black'})
display (output_widget)

<a name="step_1"></a>
## Enter the Run Parameters and Run a Simulation [&#8607;](#userguide)

### Enter the Run Parameters

- Select a margin.

  - Select `modern` for the present day margin.
  - Select `LGM` for the Last Glacial Maximum margin.
  - Select `Custom` to upload your ice margin files.

- Select the shear stress model used to calculate basil shear stress.

  - Select `Gowan` for the Gowan et al. (2021) model.
  - Select `Bedmachine` for the Bedmachine version 5 surface topography model.

- Enter the icesheet spacing.

- Enter the icesheet interval.

### Run a Simulation

- Click the `Run Simulation` button to run a simulation.


In [ ]:
# Run parameters
DROPDOWN_WIDTH = '420px'
DROPDOWN_HEIGHT = '40px'

UPLOAD_DIR = "uploaded_margins"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# File upload widget (initially hidden)
upload_widget = widgets.FileUpload(
    accept='.shp,.shx,.dbf,.prj,.cpg',
    multiple=True,
    description="Upload Margin Files",
    style={'description_width': 'initial'},  # Allow full width for the description
    layout=widgets.Layout(width='250px')   # Adjust the width of the button
)

# Status output for file validation
status_output = widgets.Output()

# Message HTML widget
message_label = widgets.HTML(
    value="<b> Note: Upload your Ice Margin files. Required file types are: .shp,.shx,.dbf,.prj,.cpg</b>",
    layout=widgets.Layout(width='auto')
)

# Function to validate and save uploaded files
def save_uploaded_files():
    required_extensions = {'.shp', '.shx', '.dbf', '.prj', '.cpg'}
    uploaded_extensions = {os.path.splitext(name)[1] for name in upload_widget.value.keys()}
    missing_extensions = required_extensions - uploaded_extensions
    
    # Clear previous files in the directory
    for file in os.listdir(UPLOAD_DIR):
        file_path = os.path.join(UPLOAD_DIR, file)
        try:
            os.remove(file_path)
        except Exception as e:
            print(f"Error deleting file {file_path}: {e}")
    
    status_output.clear_output()
    with status_output:
        if missing_extensions:
            print(f"Error: Missing files: {', '.join(missing_extensions)}")
            return
        for filename, file_info in upload_widget.value.items():
            with open(os.path.join(UPLOAD_DIR, filename), 'wb') as f:
                f.write(file_info['content'])
        print("Files uploaded and saved successfully!")

    
upload_widget.observe(lambda change: save_uploaded_files(), names='value')

# Function to dynamically display widgets
def monitor_margin_dropdown():
    prev_value = margin_dropdown.value
    while True:
        current_value = margin_dropdown.value
        if current_value != prev_value:
            clear_output(wait=True)
            # Always display other widgets and the run button
            display(
                margin_dropdown,
                shear_stress_model_dropdown,
                icesheet_spacing_integer,
                icesheet_interval_integer,
            )
            # Display the upload widget before the Run Simulation button
            if current_value.strip() == 'Custom':
                display(message_label, upload_widget, status_output)
            
            # Display the Run Simulation button last
            display(run_simulation_button)
            prev_value = current_value
        time.sleep(0.5)  # Check for changes every 0.5 seconds
margins = ['modern\xa0\xa0\xa0\xa0', 'LGM\xa0\xa0\xa0\xa0\xa0\xa0\xa0', 'Custom']
margin_dropdown = ui.Dropdown(
    name = '<h4>Margin</h4>',
    description = 'Select the margin',
    options = margins,
    value = margins[0],
    width = DROPDOWN_WIDTH,
    disabled = False
)

# Start monitoring the dropdown in the background
import threading
threading.Thread(target=monitor_margin_dropdown, daemon=True).start()

shear_stress_models = ['Gowan\xa0\xa0\xa0\xa0\xa0', 'Bedmachine']
shear_stress_model_dropdown = ui.Dropdown(
    name = '<h4>Shear Stress Model</h4>',
    description = 'Select the shear stress model',
    options = shear_stress_models,
    value = shear_stress_models[0],
    width = DROPDOWN_WIDTH,
    disabled = False
)

INTEGER_WIDTH = '441px'
INTEGER_HEIGHT = '40px'
# icesheet spacing
icesheet_spacing_integer =  ui.Integer(
    name = '<h4>Icesheet Spacing</h4>',
    description = 'Enter the icesheet spacing value',
    units = 'm',
    value = '5000',
    min = '10',
    max = '5000',
    width = INTEGER_WIDTH,
    disabled = False
)

# icesheet interval
icesheet_interval_integer =  ui.Integer(
    name = '<h4>Icesheet Interval</h4>',
    description = 'Enter the icesheet interval value',
    units = 'm',
    value = '20',
    min = '10',
    max = '100',
    width = INTEGER_WIDTH,
    disabled = False
)


In [ ]:
def run_simulation_button_callback(p):
    
    disable_widgets()
    
    run_simulation_button_callback_output.clear_output()
    results_output.clear_output()
    download_results_button_output.clear_output()
    log_output.clear_output()
    show_log_output_button.description = 'Show Log Output'
    
    plots_dir = os.path.join(Tooldir, 'plots')
    if os.path.exists(plots_dir) and os.path.isdir(plots_dir):
        shutil.rmtree(plots_dir)

    with run_simulation_button_callback_output:
        
        start_time = time.time()
        try:
            
            log_status (run_simulation_button_callback_output, 'Running a simulation...')
            
            margin = margin_dropdown.value.rstrip()
            log_info ('margin: %s' %margin)
            shear_stress_model = shear_stress_model_dropdown.value.rstrip()
            log_info ('shear_stress_model: %s' %shear_stress_model)
            icesheet_spacing = icesheet_spacing_integer.value
            log_info ('icesheet_spacing: %d' %icesheet_spacing)
            icesheet_interval = icesheet_interval_integer.value
            log_info ('icesheet_interval: %d' %icesheet_interval)
            
            # Validate custom margin files if "Custom" is selected
            if margin == 'Custom':
                required_files = {'.shp', '.shx', '.dbf', '.prj', '.cpg'}
                uploaded_files = {os.path.splitext(f)[1] for f in os.listdir(UPLOAD_DIR)}
                missing_files = required_files - uploaded_files
                if missing_files:
                    log_error(run_simulation_button_callback_output, f"Error: Missing files: {', '.join(missing_files)}")
                    enable_widgets()
                    return
                margin_arg=margin
            else:
                margin_arg=margin

            #'''
            exitStatus = subprocess.run(['./run_icesheet.sh', margin_arg, shear_stress_model, str(icesheet_spacing), str(icesheet_interval)],
                stderr=FH1, 
                stdout=FH1)
            exitCode = exitStatus.returncode
            #'''
            #exitCode = 0
            if exitCode == 0:
                
                try:
                    
                    #'''
                    # Returns PDF files containing 1 image in each PDF file
                    exitStatus = subprocess.run(['./plot.sh', margin_arg, shear_stress_model, str(icesheet_spacing), str(icesheet_interval)],
                        stderr=FH1, 
                        stdout=FH1)
                    exitCode = exitStatus.returncode
                    #'''
                    #exitCode = 0
                    if exitCode == 0:
                        pdffilepath = os.path.join(Tooldir, 'Greenland_ICESHEET_%s_%s_%dm_%dm.pdf' \
                            %(margin_arg, shear_stress_model, icesheet_spacing, icesheet_interval))
                        #print ('pdffilepath: ', pdffilepath)
                        display_results(pdffilepath, margin, shear_stress_model, icesheet_spacing, icesheet_interval)
                        display_download_results_button(pdffilepath)
                    else:
                        log_error (run_simulation_button_callback_output, "./plot.sh exit status: %s" %str(exitStatus))
                        
                except Exception as e:
                    log_error (run_simulation_button_callback_output, './plot.sh exception: %s' %str(e))
            else:
                log_error (run_simulation_button_callback_output, './run_icesheet.sh exit status: %s' %str(exitStatus))

        except Exception as e:
            log_error (run_simulation_button_callback_output, './run_icesheet.sh exception: %s' %str(e))

        elapsed_time = time.time() - start_time
        log_status (run_simulation_button_callback_output, 'Done. Elasped time: %d [s].' %round(elapsed_time))
        log_status (run_simulation_button_callback_output, 'Output files can be accessed on theghub.org in the %s directory.' %Tooldir)
        
    enable_widgets()


In [ ]:
run_simulation_button = widgets.Button(description="Run Simulation", disabled=False,\
                             layout=widgets.Layout(width=BUTTON_WIDTH, height=BUTTON_HEIGHT),\
                             style= {'button_color':'lawngreen','font_weight':'bold'})
#help (create_figures_button)
run_simulation_button.add_class("buttontextclass")

run_simulation_button.on_click (run_simulation_button_callback)
run_simulation_form = ui.Form([margin_dropdown, shear_stress_model_dropdown, icesheet_spacing_integer, icesheet_interval_integer, run_simulation_button], name = 'Run Simulation Form', width='46%')



In [ ]:
run_simulation_button_output = widgets.Output(layout={'border': widget_border_style})
display (run_simulation_button_output)
with run_simulation_button_output:
    display(run_simulation_form)

In [ ]:
run_simulation_button_callback_output = widgets.Output(layout={'border': widget_output_border_style})
display (run_simulation_button_callback_output)

<a name="step_2"></a>
## View Simulation Results [&#8607;](#userguide)

- Output from running a simulation is written to a PDF file and displayed below.<br />

- Click the `Download PDF` button to download the PDF file to your web browser.<br />


In [ ]:
import re
def natural_sort(l): 
    convert = lambda text: int(text) if text.isdigit() else text.lower()
    alphanum_key = lambda key: [convert(c) for c in re.split('([0-9]+)', key)]
    return sorted(l, key=alphanum_key)

def display_results(pdffilepath, margin, shear_stress_model, icesheet_spacing, icesheet_interval):
 
    with results_output:
        
        try:
            
            # Display images.
            # Create pdf file of the images.
            #print (type(figures))
            #print ('figures: ', figures)
            if  os.path.exists(pdffilepath):
                os.remove(pdffilepath)
            pdf = matplotlib.backends.backend_pdf.PdfPages(pdffilepath)

            # Pillow is not capable of reading pdf,
            # example, cannot identify image file './plots/base_topography.pdf' error message is returned.
            # Convert .pdf to .png
            figures = natural_sort(glob.glob('./plots' + '/*.pdf'))
            for figure in figures:

                #print ('figure: ', figure)
                figure_basename = os.path.basename(figure)
                #print ('figure_basename: ', figure_basename)
                figure_name = figure_basename[0:-4]
                #print ('figure_name: ', figure_name)
                # pdf2image.pdf2image.convert_from_path returns a list of <class 'PIL.PpmImagePlugin.PpmImageFile'> objects.
                # plots.sh is returning 1 image in each PDF file.
                image = pdf2image.pdf2image.convert_from_path(figure)[0]
                #print ('type(image): ', type(image)) #<class 'PIL.PpmImagePlugin.PpmImageFile'>
                image.save('./plots' + '/%s.png' %figure_name)            
            #''' 
            figures = natural_sort(glob.glob('./plots' + '/*.png'))
            #print ('figures: ', figures)
            num_figures = len(figures)
            #print ('num_figures: ', num_figures)
            
            header = 'Greenland ICESHEET Simulation Results'
            info = 'Margin: %s, Shear Stress Model: %s, Spacing: %d[m], Interval: %d[m]' \
                %(margin, shear_stress_model, icesheet_spacing, icesheet_interval)
            
            # Display title page
            utc_now = datetime.datetime.utcnow()
            utc_now = str(datetime.datetime(utc_now.year, utc_now.month, utc_now.day, utc_now.hour, utc_now.minute, utc_now.second))

            display(Markdown('# <center>'+header))
            display(Markdown('# <center> '))
            display(Markdown('### <center> User: %s' %User))
            display(Markdown('### <center> Date/Time UTC: %s' %utc_now))
            display(Markdown('### <center> Margin: %s' %margin))
            display(Markdown('### <center> Shear Stress Model: %s' %shear_stress_model))
            display(Markdown('### <center> Icesheet Spacing: %s' %icesheet_spacing))
            display(Markdown('### <center> Icesheet Interval: %s' %icesheet_interval))
            display(Markdown('# <center> '))

            # Fig and settings for the PDF
            fig = plt.figure()
            plt.axis('off')
            plt.text(.5, .9, header+'\n\n', va='top', ha='center', fontsize=12, fontweight='bold', wrap=False)
            plt.text(.5, .64, 'User: %s' %User, va='top', ha='center', fontsize=10, fontweight='bold', wrap=False)
            plt.text(.5, .58, 'Date/Time UTC: %s' %utc_now, va='top', ha='center', fontsize=10, fontweight='bold', wrap=False)
            plt.text(.5, .52, 'Margin: %s' %margin, va='top', ha='center', fontsize=10, fontweight='bold', wrap=False)
            plt.text(.5, .46, 'Shear Stress Model: %s' %shear_stress_model, va='top', ha='center', fontsize=10, fontweight='bold', wrap=False)
            plt.text(.5, .40, 'Icesheet Spacing: %s' %icesheet_spacing, va='top', ha='center', fontsize=10, fontweight='bold', wrap=False)
            plt.text(.5, .34, 'Icesheet Interval: %s' %icesheet_interval, va='top', ha='center', fontsize=10, fontweight='bold', wrap=False)
            pdf.savefig(fig)
            plt.close()

            for i in range(num_figures):

                figure = figures[i]
                #print ('figure: ', figure)
                figure_basename = os.path.basename(figure)
                #print ('figure_basename: ', figure_basename)
                figure_name = figure_basename[0:-4]
                #print ('figure_name: ', figure_name)
                figure_title = figure_basename
                
                figure_caption = 'fig. %d' %(i+1)
                
                image = PIL_image.open(figure) #plt_image.imread(figure)
                resized_image = image.resize((450, 700))
                #print ('type(image): ', type(image)) #<class 'PIL.PngImagePlugin.PngImageFile'>
                
                # Fig and settings for the PDF
                fig = plt.figure(figsize=(50, 50))
                ax = plt.gca()
                ax.get_xaxis().set_visible(False)
                ax.get_yaxis().set_visible(False)
                #print ('ax.axis: ', ax.axis()) #(0.0, 1.0, 0.0, 1.0)
                xmin, xmax, ymin, ymax = ax.axis()
                
                display(Markdown('## <center>'+figure_title))
                display(Markdown('### <center> '))
                HTML(display(resized_image))
                display(Markdown('### <center>'+figure_caption))
                display(Markdown('### <center> '))
                
                ax.imshow(image)
                ax.text(.5, ymax + .05, figure_title+'\n\n', va='top', ha='center', fontsize=70, fontweight='bold', wrap=False, transform=ax.transAxes)
                ax.text(.5, ymin - .105, figure_caption+'\n\n', va='bottom', ha='center', fontsize=70, fontweight='bold', wrap=False, transform=ax.transAxes)
                pdf.savefig(fig)
                plt.close()
                
            pdf.close()
            
        except Exception as e:
            
            message = 'display_results() exception encountered: %s' %str(e)
            log_status (results_output, message)


In [ ]:
def display_download_results_button(pdffilepath):
    
    with download_results_button_output:
        
        # Download file from Ghub
        display(HTML('<h3>Download %s to Your Web Browser</h3>' %os.path.basename(pdffilepath)))
        #print (os.path.relpath(self_log_filepath, Tooldir))
        downloadPDFButton = hublib.ui.Download(os.path.relpath(pdffilepath, Tooldir),
            label = 'Download PDF', style='success', icon='fa-arrow-circle-down')
        display(downloadPDFButton)
    


In [ ]:
results_output = widgets.Output(layout={'border': widget_output_border_style})
display (results_output)


In [ ]:
download_results_button_output = widgets.Output(layout={'border': widget_output_border_style})
display (download_results_button_output)

In [ ]:
# uncomment to test changes to display_results()
#results_output.clear_output()
#display_results('Greenland_ICESHEET.pdf','modern', 'Gowan', 5000, 20)

<a name="step_3"></a>
## View Log Output [&#8607;](#userguide)

- Logged output from running simulations are written to the icesheet1_log_file.txt file.<br />

- Click the `Show Log Output` button to view icesheet1_log_file.txt.

- Click the `Download File` button to download icesheet_log_file.txt to your web browser.<br />


In [ ]:
def show_log_output(change):
    
    if os.path.exists(self_log_filepath):
            
        if show_log_output_button.description == 'Show Log Output':
        
            show_log_output_button.description = 'Hide Log Output'
        
            with log_output:
            
                if os.path.exists(self_log_filepath):
                    print("%s: \n\n" %self_log_filepath)
                    f = open(self_log_filepath,'r')
                    for line in f:
                        print(line.rstrip())
                    f.close()
                else:
                    log_error (log_output, '%s does not exist ' %self_log_filepath + '. Please contact us.')
        else:
        
            show_log_output_button.description = 'Show Log Output'
            log_output.clear_output()
    else:
        log_error (log_output, '%s does not exist ' %self_log_filepath + '. Please contact us.')

show_log_output_button = widgets.Button(description="Show Log Output", disabled=False,\
    layout=widgets.Layout(width=BUTTON_WIDTH, height=BUTTON_HEIGHT),\
    style= {'button_color':'lawngreen','font_weight':'bold'})

show_log_output_button.add_class("buttontextclass")
show_log_output_button.on_click(show_log_output)
display (show_log_output_button)

In [ ]:
log_output = widgets.Output(layout={'border': widget_output_border_style})
display (log_output)

In [ ]:
# Download file from Ghub
display(HTML('<h3>Download %s to Your Web Browser</h3>' %os.path.basename(self_log_filepath)))
#print (os.path.relpath(self_log_filepath, Tooldir))
downloadTXTButton = Download(os.path.relpath(self_log_filepath, Tooldir),
    label = 'Download File', style='success', icon='fa-arrow-circle-down')
display(downloadTXTButton)